In [19]:
from geopy import distance
import folium
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

FUNÇÃO QUE RETORNA A ZONA DE RISCO DE UM FOCO DE INCENCIO COM BASE NA DISTANCIA

In [37]:
def mapa_risco(coordenadas_usuario: tuple, coordenadas_foco: tuple, horario_db: str, risco: str, idReport: int):

    media_lat = (coordenadas_usuario[0] + coordenadas_foco[0])/2
    media_lon = (coordenadas_usuario[1] + coordenadas_foco[1])/2

    mapa = folium.Map(location=[media_lat, media_lon], zoom_start=11)

    folium.Marker(
        location=coordenadas_usuario,
        popup=f"Sua Localização: {coordenadas_usuario[0]} {coordenadas_usuario[1]}",
        icon=folium.Icon(color='blue', icon='user', prefix='fa')
    ).add_to(mapa)

    folium.Marker(
        location=coordenadas_foco,
        popup=f"Foco de Incêndio\nStatus: {risco}\nCoordenadas: {coordenadas_foco[0]} {coordenadas_foco[1]}\nHorário do Foco: {horario_db}",
        icon=folium.Icon(color='red', icon='fire', prefix='fa')
    ).add_to(mapa)

    folium.Circle(
        radius=15000, # em metros
        location=coordenadas_foco,
        color='green',
        fill=True,
        fill_opacity=0.1
    ).add_to(mapa)

    # Raio Médio (Amarelo/Laranja) - 10km
    folium.Circle(
        radius=10000,
        location=coordenadas_foco,
        color='orange',
        fill=True,
        fill_opacity=0.2
    ).add_to(mapa)

    # Raio Alto (Vermelho) - 5km
    folium.Circle(
        radius=5000,
        location=coordenadas_foco,
        color='red',
        fill=True,
        fill_opacity=0.3
    ).add_to(mapa)

    mapa.save(f"../maps/mapa_report_{idReport}.html")
    

In [38]:
def risco_incendio(coordenadas_user: tuple, coordenadas_incendio: tuple, horario_relatado_str: str):

    formato = "%Y-%m-%d %H:%M:%S"
    horario_incendio = datetime.strptime(horario_relatado_str, formato)
    horario_atual = datetime.now()

    limite_tempo = timedelta(hours=3)

    if (horario_atual - horario_incendio) > limite_tempo:
        return "Expirado", 0.0

    distancia = float(distance.distance(coordenadas_user, coordenadas_incendio).km)

    if(distancia < 5.0):
        mapa_risco(coordenadas_user, coordenadas_incendio, horario_relatado_str, "Alto", 100123)
        return "Alto", distancia
    elif(distancia < 10.0):
        mapa_risco(coordenadas_user, coordenadas_incendio, horario_relatado_str, "Médio", 100123)
        return "Médio", distancia
    else:
        mapa_risco(coordenadas_user, coordenadas_incendio, horario_relatado_str, "Baixo", 100123)
        return "Baixo", distancia
    

In [39]:
# Exemplo de uso

coordenadas_foco = (-20.21016,-44.88184)
coordenadas_usuario = (-20.21016,-44.84188)

horario_db = "2026-04-01 08:04:00"

risco, distancia = risco_incendio(coordenadas_usuario, coordenadas_foco, horario_db)

print(f"Risco de incêndio: {risco}, Distância: {distancia:.2f} km")

Risco de incêndio: Alto, Distância: 4.18 km
